# 09 - Model Comparison and Error Analysis

Builds the head-to-head comparison table for fine-tuned BERT, LLM, and LLM+RAG on the same
frozen `test.csv`: accuracy, macro/weighted F1, per-class precision/recall/F1, confusion
matrices, latency, and (for the API methods) token usage and cost per document.

Identifies best accuracy, best macro F1, best minority-class (IRS) performance, fastest
method, and lowest-cost method -- without declaring one universal winner unless the
criterion is stated. Also analyzes shared errors, unique successes per method, and notable
disagreement/failure patterns.

Reads only already-saved results from `artifacts/predictions/` and `artifacts/reports/` --
it does not re-run any classifier.

### Load every method's already-saved test results -- nothing is re-run here

**Purpose:** Load the frozen test split (for looking up document text during error
analysis) and the saved metrics reports and row-level predictions for all three methods
(BERT, LLM, LLM+RAG).

**Why this step is necessary:** This notebook's entire job is to *compare* results that
already exist -- it deliberately never calls a classifier or makes an API call. Reading back
the exact files saved by notebooks 05, 06, and 08 (rather than recomputing anything) is what
guarantees the comparison reflects each method's one true frozen test evaluation, with
nothing recalculated differently here.

**Inputs:** `data/splits/test.csv`, and `artifacts/reports/{bert,llm,llm_rag}_test_metrics.json`
plus `artifacts/predictions/{bert,llm,llm_rag}_test.json`.

**Output:** `reports` (a dict of `MetricsReport` objects, one per method) and `predictions`
(a dict of dicts, mapping method -> document_id -> that method's prediction for that
document).

**How to interpret the result:** No printed output here; success just means every method's
saved results loaded without error, ready for side-by-side comparison below.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..") / "src"))

import pandas as pd

from newstart_ai.config import load_settings
from newstart_ai.data import load_split
from newstart_ai.evaluation import load_metrics_report, load_predictions

settings = load_settings()
train_df, val_df, test_df, manifest = load_split(settings)
ds_cfg = settings.base.dataset

methods = ["bert", "llm", "llm_rag"]
# Every number and prediction below comes from files saved by notebooks 05/06/08 -- nothing
# in this notebook re-runs a classifier or calls an external API.
reports = {m: load_metrics_report(m, "test", settings) for m in methods}
predictions = {m: {r.document_id: r for r in load_predictions(m, "test", settings)} for m in methods}

## Summary comparison table

### Build the head-to-head summary table

**Purpose:** Lay each method's headline metrics -- accuracy, macro/weighted precision,
recall, F1, mean latency, total cost, and cost per document -- side by side in one table.

**Why this step is necessary:** This is the single table that answers "how do the three
methods compare overall?" Having all three methods' numbers in one place, in the same units,
is what makes the criteria-based comparison a few cells down possible.

**Inputs:** `reports` (loaded above).

**Output:** `summary`, a DataFrame with one row per method.

**How to interpret the result:** Macro F1 is this project's primary metric -- look there
first, not at accuracy alone. BERT has no `total_estimated_cost`/`cost_per_document` (shown
as missing/NaN) because it runs locally after training, with no per-call API cost, unlike
the two LLM-based methods.

In [2]:
summary = pd.DataFrame(
    [
        {
            "method": m,
            "accuracy": reports[m].accuracy,
            "macro_precision": reports[m].macro_precision,
            "macro_recall": reports[m].macro_recall,
            "macro_f1": reports[m].macro_f1,
            "weighted_f1": reports[m].weighted_f1,
            "mean_latency_ms": reports[m].mean_latency_ms,
            "total_estimated_cost": reports[m].total_estimated_cost,
            "cost_per_document": reports[m].cost_per_document,
        }
        for m in methods
    ]
)
summary

,method,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,mean_latency_ms,total_estimated_cost,cost_per_document
0,bert,0.993377,0.958333,0.993750,0.974108,0.993637,16.419130,NaN,NaN
1,llm,0.986755,0.953526,0.989205,0.969387,0.987016,1507.914583,0.109898,0.000728
2,llm_rag,0.986755,0.953526,0.989205,0.969387,0.987016,1555.752865,0.123073,0.000815


## Per-class comparison (all methods, all four agencies)

Macro F1 is the primary metric throughout this project -- per-class numbers are always shown alongside it so IRS's small test slice (5 documents) is never hidden behind an aggregate.

### Compare per-class F1 across methods

**Purpose:** Reshape each method's per-class metrics into a table with one row per agency
and one column per method, focused on F1.

**Why this step is necessary:** The summary table above hides how each method does on any
one specific agency behind an aggregate macro F1 number. This view makes it possible to spot,
for example, that IRS's F1 is uniformly lower across all three methods -- something the
aggregate numbers alone wouldn't reveal.

**Inputs:** `reports[m].per_class` for each method `m`.

**Output:** `per_class_df` (a long-format DataFrame) and a pivoted view with agencies as
rows and methods as columns.

**How to interpret the result:** Any single low cell here is worth investigating -- in this
project's run, IRS is visibly lower than the other three agencies across every method, which
is exactly the small-sample effect flagged since notebook 03.

In [3]:
per_class_rows = []
for m in methods:
    for pc in reports[m].per_class:
        per_class_rows.append({"method": m, **pc.model_dump()})
per_class_df = pd.DataFrame(per_class_rows)
per_class_df.pivot(index="label", columns="method", values="f1")

method,bert,llm,llm_rag
label,,,
DMV,1.000000,0.990826,0.990826
IRS,0.909091,0.909091,0.909091
SSA,0.987342,0.987342,0.987342
USCIS,1.000000,0.990291,0.990291


### Zoom in on IRS specifically

**Purpose:** Show IRS's precision, recall, F1, and support (row count) for every method in
one small table.

**Why this step is necessary:** IRS is this dataset's smallest class by a wide margin, and
its test-set support (5 documents) is small enough that a single misclassification swings
its precision or recall dramatically. Printing the caveat directly above the numbers, every
time IRS is shown, is a deliberate choice so the small sample size is never separated from
the metric itself.

**Inputs:** `per_class_df`, filtered to IRS.

**Output:** A small DataFrame with one row per method.

**How to interpret the result:** A precision below 1.0 despite recall of 1.0 (as seen here)
means every true IRS document was correctly found, but at least one non-IRS document was
also (incorrectly) labeled IRS -- with only 5 IRS documents in the denominator, even one such
mistake has an outsized effect on the precision number.

In [4]:
print("IRS per-class detail (small-sample caveat applies -- 5 test documents):")
per_class_df[per_class_df["label"] == "IRS"][["method", "precision", "recall", "f1", "support"]]

IRS per-class detail (small-sample caveat applies -- 5 test documents):


,method,precision,recall,f1,support
3,bert,0.833333,1.0,0.909091,5
7,llm,0.833333,1.0,0.909091,5
11,llm_rag,0.833333,1.0,0.909091,5


## Confusion matrices

### Print each method's confusion matrix

**Purpose:** Show, for every method, exactly which true label got predicted as which label
-- not just the aggregate error rate.

**Why this step is necessary:** A confusion matrix reveals *what kind* of mistake a method
makes, not just how many it makes. This is what makes it possible to notice, a few cells
below, that the exact same two documents account for essentially all of the errors across
all three methods -- information a single accuracy number could never reveal.

**Inputs:** `reports[m].confusion_matrix` and `reports[m].confusion_matrix_labels` for each
method.

**Output:** Printed text -- one small table per method, rows and columns both labeled with
the four agency names.

**How to interpret the result:** The diagonal holds correct predictions; anything off the
diagonal is a specific misclassification (row = true label, column = predicted label).
Compare the three matrices to each other -- in this project's run, BERT's matrix has one
off-diagonal entry, while the LLM and LLM+RAG matrices (which turn out to be identical to
each other) each have two.

In [5]:
for m in methods:
    print(f"--- {m} ---")
    print(pd.DataFrame(
        reports[m].confusion_matrix,
        index=reports[m].confusion_matrix_labels,
        columns=reports[m].confusion_matrix_labels,
    ))
    print()

--- bert ---
       USCIS  DMV  SSA  IRS
USCIS     51    0    0    0
DMV        0   55    0    0
SSA        0    0   39    1
IRS        0    0    0    5

--- llm ---
       USCIS  DMV  SSA  IRS
USCIS     51    0    0    0
DMV        1   54    0    0
SSA        0    0   39    1
IRS        0    0    0    5

--- llm_rag ---
       USCIS  DMV  SSA  IRS
USCIS     51    0    0    0
DMV        1   54    0    0
SSA        0    0   39    1
IRS        0    0    0    5



## Criteria-based comparison

Each "winner" below is only the best method *for that specific criterion* -- this project
does not declare one universal winner.

### Identify the best method per criterion -- no single overall winner declared

**Purpose:** Pull out, from `summary` and `per_class_df`, which method is best on each of
several different criteria: overall accuracy, macro F1, IRS F1 specifically, latency, and
cost per document.

**Why this step is necessary:** The project's design explicitly avoids declaring one
universal "winner" -- different users care about different tradeoffs (a batch job might
care most about accuracy; an interactive demo might care most about latency). Presenting
each criterion's winner separately, and stating BERT's zero marginal cost explicitly, gives
a reader the information needed to make that tradeoff themselves rather than being handed a
single, oversimplified verdict.

**Inputs:** `summary` and `per_class_df` (both computed above).

**Output:** Printed text identifying the best method for each criterion.

**How to interpret the result:** In this project's run BERT wins on accuracy, macro F1, IRS
F1, and speed -- but the plain LLM has the lowest per-document API cost of the two
LLM-based methods, and BERT's "cost" only appears free because its (one-time) training cost
already happened in notebook 04.

In [6]:
best_accuracy = summary.loc[summary["accuracy"].idxmax()]
best_macro_f1 = summary.loc[summary["macro_f1"].idxmax()]
best_irs_f1 = per_class_df[per_class_df["label"] == "IRS"].loc[per_class_df[per_class_df["label"] == "IRS"]["f1"].idxmax()]
fastest = summary.loc[summary["mean_latency_ms"].idxmin()]
api_costs = summary.dropna(subset=["cost_per_document"])
lowest_cost = api_costs.loc[api_costs["cost_per_document"].idxmin()] if not api_costs.empty else None

print(f"Best accuracy:        {best_accuracy['method']} ({best_accuracy['accuracy']:.4f})")
print(f"Best macro F1:        {best_macro_f1['method']} ({best_macro_f1['macro_f1']:.4f})")
print(f"Best IRS F1:          {best_irs_f1['method']} ({best_irs_f1['f1']:.4f}) -- small-sample caveat applies")
print(f"Fastest:              {fastest['method']} ({fastest['mean_latency_ms']:.1f} ms/doc)")
if lowest_cost is not None:
    print(f"Lowest cost/document: {lowest_cost['method']} (${lowest_cost['cost_per_document']:.6f}/doc)")
print(f"BERT has no per-document API cost (runs locally on GPU) -- cost/accuracy tradeoffs should weigh that.")

Best accuracy:        bert (0.9934)
Best macro F1:        bert (0.9741)
Best IRS F1:          bert (0.9091) -- small-sample caveat applies
Fastest:              bert (16.4 ms/doc)
Lowest cost/document: llm ($0.000728/doc)
BERT has no per-document API cost (runs locally on GPU) -- cost/accuracy tradeoffs should weigh that.


## Error analysis

Across all 151 test documents and all three methods, only two documents were ever
misclassified by any method. Both are worth reading individually rather than treated as pure
model failures.

### Find every document any method got wrong

**Purpose:** Scan all three methods' test-set predictions and collect the set of document
IDs where at least one method's prediction didn't match the true label, then build a table
showing what each method predicted for those specific documents.

**Why this step is necessary:** Rather than treating "errors" as an abstract count, this
step identifies the *exact* documents behind every mistake, across all three methods at
once -- which is what makes it possible to read the actual text of those documents next and
determine whether the mistake was really the model's fault.

**Inputs:** `predictions` (each method's document_id -> prediction mapping).

**Output:** `misclassified_ids` (a set of document IDs) and `error_df` (a table showing the
true label and each method's prediction for those specific documents).

**How to interpret the result:** In this project's run, only two documents (out of 151) were
ever misclassified by any method -- an unusually small error set, which is exactly why it's
worth reading those two documents individually rather than just reporting an error rate.

In [7]:
misclassified_ids = set()
for m in methods:
    for doc_id, pred in predictions[m].items():
        if pred.predicted_label != pred.true_label:
            misclassified_ids.add(doc_id)

rows = []
for doc_id in misclassified_ids:
    row = {"document_id": doc_id, "true_label": predictions["bert"][doc_id].true_label}
    for m in methods:
        row[f"{m}_predicted"] = predictions[m][doc_id].predicted_label
    rows.append(row)
error_df = pd.DataFrame(rows)
error_df

,document_id,true_label,bert_predicted,llm_predicted,llm_rag_predicted
0,533,DMV,DMV,USCIS,USCIS
1,541,SSA,IRS,IRS,IRS


### Read the actual text of the misclassified documents

**Purpose:** Print the first 300 characters of each misclassified document's actual text,
along with its labeled agency and form number.

**Why this step is necessary:** A confusion matrix and an error table show *that* a mistake
happened, but not *why*. Reading the real document text is what turns "the model was wrong"
into an actual, checkable explanation -- in this project's case, revealing that both errors
trace back to problems with the source data itself (a likely mislabel and a failed text
extraction), not a weakness in any of the three classification methods.

**Inputs:** `misclassified_ids` and `test_df` (to look up each document's full text).

**Output:** Printed text -- one excerpt per misclassified document.

**How to interpret the result:** See the two markdown notes directly below this cell's
output for the specific conclusion drawn from each document's text.

In [8]:
for doc_id in misclassified_ids:
    row = test_df[test_df[ds_cfg.id_column].astype(str) == doc_id].iloc[0]
    print(f"--- document_id={doc_id}  labeled={row[ds_cfg.label_column]}  form_number={row.get('form_number')} ---")
    print(row[ds_cfg.text_column][:300].replace(chr(10), " "))
    print()

--- document_id=533  labeled=DMV  form_number=nan ---
 Get Adobe Reader Now!

--- document_id=541  labeled=SSA  form_number=nan ---
 (Rev. diciembre de 2025) agencias gubernamentales, entidades de tribus indígenas de los EE. UU



**Document 541** is labeled `SSA` in the dataset, but its text is clearly IRS Form SS-4
("Solicitud de Numero de Identificacion del Empleador (EIN)" -- Application for Employer
Identification Number, Department of the Treasury). All three methods independently
predicted `IRS`. This looks like a **dataset labeling error carried over from
`00_data_acquisition`** -- form SS-4 is filed with the IRS despite its "SS" prefix -- not a
model failure. It's the one error BERT, LLM, and LLM+RAG all share.

**Document 533** is labeled `DMV`, but its extracted text is only a generic PDF-portfolio
placeholder message ("open this PDF portfolio in Acrobat..."), with no real document content
at all. This is an **upstream text-extraction failure**, not a labeling or model error -- the
LLM and LLM+RAG both guessed `USCIS` with no real signal to go on; BERT happened to predict
the correct label `DMV`, most likely coincidentally given the near-empty input.

Per docs/BLUEPRINT.md and the dataset-validation policy, both should be **fixed upstream**
(re-labeled and re-extracted respectively) in `00_data_acquisition`, producing a new
documented dataset version -- not silently patched here. Recorded as a limitation in
`10_research_summary.ipynb` instead.

## LLM+RAG vs LLM: no measurable difference on this test set

LLM and LLM+RAG produced **identical predictions on all 151 test documents** (same
confusion matrix, same two errors). Retrieval context did not change a single outcome here.
Given the EDA finding that each agency's vocabulary is already highly distinctive
(`02_exploratory_data_analysis.ipynb`), the classification task may simply be easy enough
that additional retrieved context has no headroom left to improve -- a genuine finding, not
a bug, and worth stating plainly rather than assuming RAG "must" help.

## Save the comparison artifacts for the research summary

### Save the comparison tables and error notes for the research summary

**Purpose:** Write the summary comparison table, the per-class table, the error-analysis
table, and a short written explanation of each error, all to `artifacts/reports/`.

**Why this step is necessary:** Notebook 10 (the research summary) and, eventually, the demo
app's Research Results page are meant to *read* these files rather than recomputing this
analysis -- saving them here is what makes this notebook's findings reusable downstream
without re-deriving them.

**Inputs:** `summary`, `per_class_df`, and `error_df` (all built earlier in this notebook).

**Output:** Four files under `artifacts/reports/`: `model_comparison_summary.csv`,
`model_comparison_per_class.csv`, `error_analysis.csv`, and `error_analysis_notes.json`.

**How to interpret the result:** The printed path confirms where these files landed; their
contents are exactly what's shown in the tables above, just persisted for reuse.

In [9]:
import json

comparison_dir = settings.resolve_path("artifacts/reports")
comparison_dir.mkdir(parents=True, exist_ok=True)

summary.to_csv(comparison_dir / "model_comparison_summary.csv", index=False)
per_class_df.to_csv(comparison_dir / "model_comparison_per_class.csv", index=False)
error_df.to_csv(comparison_dir / "error_analysis.csv", index=False)

# A short, human-readable explanation for each specific misclassified document -- written
# once here so notebook 10 and the future Research Results page don't have to re-derive
# "why" from the raw predictions alone.
with open(comparison_dir / "error_analysis_notes.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "541": "Labeled SSA but text is IRS Form SS-4 (EIN application) -- likely a dataset labeling error, not a model failure. All three methods predicted IRS.",
            "533": "Labeled DMV but extracted text is an empty PDF-portfolio placeholder -- an upstream extraction failure. LLM and LLM+RAG guessed USCIS with no real signal; BERT happened to predict DMV correctly.",
        },
        f,
        indent=2,
    )

print(f"Saved comparison artifacts to {comparison_dir}")

Saved comparison artifacts to D:\USD\Projects\a590\newstart-ai\newstart_ai_benchmark\artifacts\reports


## Summary for the next notebook

- BERT: highest macro F1 (fewest errors: 1/151), fastest, free per-inference.
- LLM and LLM+RAG: tied, both macro F1 slightly below BERT, identical predictions to each
  other -- RAG added cost and latency without changing any outcome on this test set.
- Both test-set errors trace to upstream data-quality issues (one mislabel, one empty
  extraction), not model weaknesses -- flagged for `00_data_acquisition`, not silently fixed.
- Next: `10_research_summary.ipynb` produces the final research-ready write-up.